In [43]:
# Import necessary libraries for data manipulation, visualization, machine learning, and deep learning.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import Perceptron    # Used for simple linear classification tasks.

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential     # Sequential lets you build a neural network layer-by-layer in Keras.

from tensorflow.keras.layers import Dense     #Dense makes the final predictions
from tensorflow.keras.layers import Conv2D     # Conv2D extracts features
from tensorflow.keras.layers import Flatten    # Flatten reshapes them

from tensorflow.keras.layers import MaxPooling2D     # MaxPooling2D reduces size
from tensorflow.keras.layers import Dropout          # Dropout prevents overfitting

from tensorflow.keras.utils import to_categorical     # converts numeric class labels into one-hot encoded format for training classification models

In [44]:
# Load the training and testing datasets from CSV files into pandas DataFrames.
df = pd.read_csv("mnist_train.csv")
df_test = pd.read_csv("mnist_test.csv")

In [72]:
df

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
# Display the first few rows of the training DataFrame to get a glimpse of the data structure.
df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [46]:
# Check the dimensions (number of rows and columns) of the training DataFrame.
df.shape

(60000, 785)

In [47]:
# Display the names of all columns in the training DataFrame.
df.columns

Index(['label', '1x1', '1x2', '1x3', '1x4', '1x5', '1x6', '1x7', '1x8', '1x9',
       ...
       '28x19', '28x20', '28x21', '28x22', '28x23', '28x24', '28x25', '28x26',
       '28x27', '28x28'],
      dtype='object', length=785)

In [75]:
df_test.columns


Index(['label', '1x1', '1x2', '1x3', '1x4', '1x5', '1x6', '1x7', '1x8', '1x9',
       ...
       '28x19', '28x20', '28x21', '28x22', '28x23', '28x24', '28x25', '28x26',
       '28x27', '28x28'],
      dtype='object', length=785)

In [48]:
# Get a concise summary of the training DataFrame, including data types and non-null values.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to 28x28
dtypes: int64(785)
memory usage: 359.3 MB


In [49]:
# Calculate the sum of null (missing) values for each column in the training DataFrame.
df.isnull().sum()

,0
label,0
1x1,0
1x2,0
1x3,0
1x4,0
...,...
28x24,0
28x25,0
28x26,0
28x27,0


In [50]:
# Preprocessing step: Separate features (X) and labels (y) for both training and test datasets.
# X_train: Features for training (all columns except 'label'), converted to a NumPy array.
# y_train: Labels for training (only the 'label' column), converted to a NumPy array.
# X_test: Features for testing (all columns except 'label') from the test DataFrame, converted to a NumPy array.
# y_test: Labels for testing (only the 'label' column) from the test DataFrame, converted to a NumPy array.
# Note: In some real-world scenarios, df_test might not have a 'label' column, in which case y_test would not be created.
#preprocess
X_train = df.drop("label", axis=1).values
y_train = df["label"].values
X_test = df_test.drop("label", axis=1).values
y_test = df_test["label"].values

In [51]:
# Normalize the pixel values of the training and test features.
# Pixel values are typically between 0 and 255. Dividing by 255.0 scales them to a range of 0 to 1,
# which is beneficial for neural network training.
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [76]:
# Reshape the flattened 1D pixel arrays into 2D images (28x28 pixels).
# This is often required for models that process images in a 2D format, like CNNs, or for visualization.
X_train_img = X_train.reshape(-1, 28, 28)
X_test_img = X_test.reshape(-1, 28, 28)

In [77]:
X_train_img

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.

In [53]:
# Convert the integer class labels into one-hot encoded vectors.
# For example, a label '5' becomes [0, 0, 0, 0, 0, 1, 0, 0, 0, 0].
# This is necessary for training neural networks with a 'categorical_crossentropy' loss function.
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [54]:
# Define a simple Perceptron model (single-layer neural network).
# It flattens the 28x28 input image and then uses a single Dense layer with a softmax activation
# for multi-class classification (10 classes for digits 0-9).
perceptron = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(10, activation="softmax")
])

In [55]:
# Compile the Perceptron model.
# 'sgd' (Stochastic Gradient Descent) is the optimizer.
# 'categorical_crossentropy' is the loss function, suitable for multi-class classification with one-hot encoded labels.
# 'accuracy' is used as a metric to monitor performance during training.
perceptron.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

In [57]:
# Train the Perceptron model using the training data.
# 'epochs=5': The model will iterate over the entire training dataset 5 times.
# 'batch_size=32': The model will update its weights after processing 32 samples.
# 'validation_data': Uses the test data to evaluate the model's performance on unseen data after each epoch.
# 'verbose=1': Displays a progress bar during training.
history_percp = perceptron.fit(X_train_img, y_train_cat, epochs=5, batch_size=32, validation_data=(X_test_img, y_test_cat), verbose=1)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9041 - loss: 0.3475 - val_accuracy: 0.9100 - val_loss: 0.3274
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9057 - loss: 0.3387 - val_accuracy: 0.9142 - val_loss: 0.3203
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9080 - loss: 0.3317 - val_accuracy: 0.9133 - val_loss: 0.3150
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9089 - loss: 0.3259 - val_accuracy: 0.9150 - val_loss: 0.3114
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9104 - loss: 0.3209 - val_accuracy: 0.9156 - val_loss: 0.3069


In [59]:
# Evaluate the trained Perceptron model on the test dataset.
# 'verbose=0' suppresses the output during evaluation.
# [1] extracts the accuracy score from the evaluation results.
acc_percp = perceptron.evaluate(X_test_img, y_test_cat, verbose=0)[1]

In [60]:
# Display the accuracy achieved by the Perceptron model on the test set.
acc_percp

0.9156000018119812

In [61]:
# Define an Artificial Neural Network (ANN) model.
# It consists of an input Flatten layer, followed by two Dense (fully connected) hidden layers with 'relu' activation,
# and a final Dense output layer with 'softmax' activation for 10-class classification.
#ANN
ann = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

In [62]:
# Compile the ANN model.
# 'adam' is chosen as the optimizer, which is an adaptive learning rate optimization algorithm.
# 'categorical_crossentropy' is the loss function, and 'accuracy' is the metric.
ann.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [63]:
# Train the ANN model using the training data, similar to the Perceptron.
# 'epochs=5', 'batch_size=32', and 'validation_data' are configured for training and monitoring performance.
history_ann = ann.fit(X_train_img, y_train_cat, epochs=5, batch_size=32, validation_data=(X_test_img, y_test_cat), verbose=1)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9316 - loss: 0.2340 - val_accuracy: 0.9543 - val_loss: 0.1409
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9696 - loss: 0.0985 - val_accuracy: 0.9710 - val_loss: 0.0908
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9779 - loss: 0.0706 - val_accuracy: 0.9745 - val_loss: 0.0896
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9833 - loss: 0.0528 - val_accuracy: 0.9792 - val_loss: 0.0779
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9859 - loss: 0.0432 - val_accuracy: 0.9718 - val_loss: 0.0988


In [64]:
# Evaluate the trained ANN model on the test dataset to get its accuracy.
acc_ann = ann.evaluate(X_test_img, y_test_cat, verbose=0)[1]

In [65]:
# Display the accuracy achieved by the ANN model on the test set.
acc_ann

0.9718000292778015

In [66]:
# Reshape the image data for Convolutional Neural Networks (CNNs).
# CNNs typically expect input in the format (batch_size, height, width, channels).
# For grayscale images like MNIST, channels=1.
X_train_cnn = X_train.reshape(-1, 28, 28,1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [67]:
# Define a Convolutional Neural Network (CNN) model.
# It includes Conv2D layers for feature extraction, MaxPooling2D layers for dimensionality reduction,
# a Flatten layer to convert 2D features to 1D, Dense layers for classification, and Dropout for regularization.
cnn = Sequential([
    Conv2D(32, kernel_size=(3,3), activation="relu", input_shape=(28,28,1)),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(64, kernel_size=(3,3), activation="relu"),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10, activation="softmax")
])

In [68]:
# Compile the CNN model using the 'adam' optimizer, 'categorical_crossentropy' loss, and 'accuracy' metric.
cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [69]:
# Train the CNN model using the reshaped training data.
# Training parameters like epochs, batch size, and validation data are set.
history_cnn = cnn.fit(X_train_cnn, y_train_cat, epochs=5, batch_size=32, validation_data=(X_test_cnn, y_test_cat), verbose=1)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 31ms/step - accuracy: 0.9321 - loss: 0.2223 - val_accuracy: 0.9835 - val_loss: 0.0519
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 31ms/step - accuracy: 0.9742 - loss: 0.0864 - val_accuracy: 0.9878 - val_loss: 0.0365
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 83s 31ms/step - accuracy: 0.9821 - loss: 0.0616 - val_accuracy: 0.9895 - val_loss: 0.0321
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9847 - loss: 0.0498 - val_accuracy: 0.9910 - val_loss: 0.0269
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 58s 31ms/step - accuracy: 0.9872 - loss: 0.0409 - val_accuracy: 0.9907 - val_loss: 0.0304


In [70]:
# Evaluate the trained CNN model on the test dataset to obtain its accuracy.
acc_cnn = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)[1]

In [71]:
# Display the accuracy achieved by the CNN model on the test set.
acc_cnn

0.9907000064849854